# Research pilot — simple neural network versus small Transformer

**You start the experiment.** This notebook uses separate research models and folders. Your existing player, checkpoints and current run are preserved.

**First chunk: 128 training games per model (256 total), in 64-game learning batches.** Models run one after the other, using two CPU threads and no parallel game workers. There is a **90-minute soft limit per launch** checked between games; a final game or learning update can overrun it. Expect approximately **30–90 minutes**, to be refined by the first actual run. This is a feasibility and timing pilot, not evidence that one architecture wins.

Both models start fresh with the same initialization seed, encoder, legal actions, eleven-class deck roster, PPO settings and common frozen saved-Transformer opponent. No tactical warm-up or handwritten opponent is used in this first controlled comparison. Each model gets 44 baseline and 44 post-chunk evaluation games with a shared held-out game-seed schedule. Those cases reuse the deck roster and opponent; they do not test unseen-deck generalization. One training seed and 44 evaluation games have substantial uncertainty.

**Let the existing pilot/long run finish first.** The launcher refuses overlap. Run All Cells checks configuration and then launches one bounded comparison chunk. If the soft limit pauses it, Run All again resumes the unfinished paired round, including saved partial-batch samples, rather than granting another full chunk. Once both arms finish, a later launch adds another 128 games to each. Manual interruption can lose unsaved games since the last checkpoint; the soft-limit pause saves them.

Results live in `runs/research_mlp_transformer`, entirely separate from player development. The dashboard refreshes after each learning batch, with evaluation status at most every 15 seconds. Larger model sizes, recurrent memory, search, multiple seeds and unseen-deck testing are later study phases.

**Readiness check:** the pilot’s two new cost-modifier/Deathrattle-context errors are repaired. The study is pinned to the repaired engine and the unchanged 2,920-game opponent. Engine and research checks plus preflight passed. No experiment has been started.


In [ ]:
from pathlib import Path
import json, subprocess, sys, time, html
from IPython.display import HTML, display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'tools/run_architecture_comparison.py').exists())
command=[sys.executable,str(ROOT/'tools/run_architecture_comparison.py')]
# No games, evaluation or weight updates in this cell.
check=subprocess.run(command,cwd=ROOT,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(check.stdout)
PREFLIGHT_OK=check.returncode==0
if not PREFLIGHT_OK:raise RuntimeError('Research checks failed; experiment will not start.')


In [ ]:
# Running this cell is YOUR explicit start. It launches at most one bounded paired round.
if not globals().get('PREFLIGHT_OK',False):raise RuntimeError('Run the successful checks first.')
rows={name:dict(games='—',score='—',status='Waiting') for name in ('mlp','transformer')}
phase='Starting';last_draw=0.;begin=time.monotonic()
def render():
    table=''.join('<tr><td>'+html.escape(name)+'</td><td>'+str(r['games'])+'</td><td>'+html.escape(r['score'])+'</td><td>'+html.escape(r['status'])+'</td></tr>' for name,r in rows.items())
    return '<div style="padding:18px;border:1px solid #888;border-radius:12px"><h3>Architecture comparison</h3><p>'+html.escape(phase)+'</p><p>Elapsed: '+str(round((time.monotonic()-begin)/60,1))+' minutes · 90-minute soft limit</p><table><tr><th>Model</th><th>Training games</th><th>Evaluation score</th><th>Status</th></tr>'+table+'</table><p>Score = win + half a draw. New game seeds, same deck roster/opponent. This small pilot cannot establish an architecture ranking.</p></div>'
screen=display(HTML(render()),display_id=True)
with subprocess.Popen(command+['--train'],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as process:
    try:
        for line in process.stdout:
            try:event=json.loads(line)
            except ValueError:print(line.rstrip());continue
            kind=event.get('event');name=event.get('architecture')
            if name in rows:
                rows[name]['games']=event.get('games',rows[name]['games'])
                if kind=='research_evaluation':rows[name]['status']='Evaluation '+str(event['done'])+'/'+str(event['total'])+' (no learning)'
                elif kind=='research_batch':rows[name]['status']='Learning batch saved';phase='Training: '+name
                elif kind=='research_arm_complete':
                    score=event['evaluation']['score'];rows[name]['score']='—' if score is None else format(score,'.1%');rows[name]['status']='Chunk complete'
            if kind=='research_paused':phase='Paused: '+event['reason']+' — Run All resumes this round'
            elif kind=='research_round_complete':phase='Both models finished equal training budgets. Experiment stopped.'
            now=time.monotonic()
            if kind!='research_evaluation' or now-last_draw>=15:
                screen.update(HTML(render()));last_draw=now
            if kind!='research_evaluation':print(json.dumps(event))
        code=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=30)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
if code:raise RuntimeError('Experiment stopped. Read the error; earlier checkpoints remain intact.')
